# Lesson 5A: Embeddings & Semantic Search (RAG Basics)

In this lesson, you'll learn how to use embeddings for semantic search and build a basic RAG system.

## Topics Covered
1. Understanding embeddings
2. Creating and using embeddings
3. Semantic similarity and search
4. Building a simple vector store
5. Basic RAG (Retrieval Augmented Generation)

## Learning Objectives
- Understand what embeddings are and how they work
- Create embeddings for text
- Measure similarity between texts
- Build a searchable knowledge base
- Implement basic RAG to answer questions from in-memory documents

In [ ]:
# Install required packages if not already installed
#%pip install python-dotenv
#%pip install openai
#%pip install numpy

# Load environment variables from .env file
import os
from dotenv import load_dotenv
load_dotenv()
import openai
import numpy as np
from typing import List, Dict
print("OpenAI package version:", openai.__version__)
print("NumPy version:", np.__version__)

In [ ]:
# Set up the OpenAI client with environment variables
chat_client = openai.OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=int(os.getenv("OPENAI_TIMEOUT", 30)),
    max_retries=int(os.getenv("MAX_RETRIES", 3)),
    base_url=os.getenv("OPENAI_ENDPOINT")    
)

print("Client configured successfully!")

## 1. Understanding Embeddings

**What are embeddings?**
Embeddings convert text into numbers (vectors) that capture meaning.

**Analogy:**
- **GPS coordinates** for locations → **Embeddings** for meaning
- Cities close together geographically have similar coordinates
- Words/phrases with similar meanings have similar embeddings

**Key concept:**
- "dog" and "puppy" have similar embeddings (close meaning)
- "dog" and "car" have different embeddings (different meaning)

**Why embeddings matter:**
- Enable semantic search (search by meaning, not just keywords)
- Power RAG systems (find relevant context for AI)
- Enable clustering, classification, recommendations

In [ ]:
# Example 1A: Creating your first embedding
def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding for a text string"""
    text = text.replace("\n", " ")  # Clean text
    response = chat_client.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

# Get embedding for a simple sentence
text = "The cat sat on the mat"
embedding = get_embedding(text)

print(f"Text: {text}")
print(f"\nEmbedding dimensions: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")
print(f"\n✅ Embedding is a list of {len(embedding)} numbers representing the meaning!")

In [ ]:
# Example 1B: Embeddings for different texts
texts = [
    "I love pizza",
    "Pizza is my favorite food",
    "The weather is sunny today",
    "Python is a programming language"
]

print("Creating embeddings for multiple texts...\n")
embeddings = {}

for text in texts:
    embedding = get_embedding(text)
    embeddings[text] = embedding
    print(f"✓ '{text[:40]}...' → {len(embedding)} dimensions")

print(f"\n📊 Created {len(embeddings)} embeddings!")
print("\nKey insight: Each text becomes a point in {}-dimensional space".format(len(embedding)))

## 2. Semantic Similarity

**Cosine similarity** measures how similar two embeddings are.

**Range:**
- `1.0` = Identical meaning
- `0.8-0.9` = Very similar
- `0.5-0.7` = Somewhat related
- `< 0.5` = Different meanings

**Analogy:** Like measuring the angle between two arrows:
- Same direction (similar meaning) = high similarity
- Opposite directions (different meaning) = low similarity

In [ ]:
# Example 2A: Calculate cosine similarity
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    return dot_product / (norm1 * norm2)

# Compare similar sentences
text1 = "I love dogs"
text2 = "I adore puppies"
text3 = "The weather is cold"

# Step 1: Get the embedding for a text
emb1 = get_embedding(text1)
emb2 = get_embedding(text2)
emb3 = get_embedding(text3)

# Step 2: perform cosine similarity between embeddings 
# Internally a vector dot product is created that is normalized by the magnitudes of the vectors to give a similarity score between -1 and 1)
similarity_12 = cosine_similarity(emb1, emb2)
similarity_13 = cosine_similarity(emb1, emb3)

print("Comparing text similarities:\n")
print(f"Text 1: '{text1}'")
print(f"Text 2: '{text2}'")
print(f"Similarity: {similarity_12:.3f} (Similar meaning!)\n")

print(f"Text 1: '{text1}'")
print(f"Text 3: '{text3}'")
print(f"Similarity: {similarity_13:.3f} (Different meaning!)\n")

print("✅ Higher similarity = more similar meaning")

In [ ]:
# Example 2B: Finding most similar text
def find_most_similar(query, texts_with_embeddings):
    """Find the most similar text to a query"""
    query_embedding = get_embedding(query)
    
    similarities = []
    for text, embedding in texts_with_embeddings.items():
        similarity = cosine_similarity(query_embedding, embedding)
        similarities.append((text, similarity))
    
    # Sort by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

# Create a small knowledge base
knowledge = {
    "Python is a programming language used for web development and data science.": None,
    "JavaScript is used for building interactive websites.": None,
    "Machine learning helps computers learn from data.": None,
    "The Eiffel Tower is located in Paris, France.": None,
    "Pizza originated in Italy and is now popular worldwide.": None
}

# Generate embeddings
print("Building knowledge base...\n")
for text in knowledge.keys():
    knowledge[text] = get_embedding(text)

# Test queries
queries = [
    "What programming language should I learn?",
    "Tell me about Italian food",
    "How does AI work?"
]

for query in queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}\n")
    results = find_most_similar(query, knowledge)
    
    print("Top 3 matches:")
    for i, (text, score) in enumerate(results[:3], 1):
        print(f"  {i}. [{score:.3f}] {text[:60]}...")

## 3. Building a Simple Vector Store

A **vector store** is a database for embeddings that enables fast semantic search.

**Components:**
1. Documents (text chunks)
2. Embeddings (vector representations)
3. Metadata (optional info about each document)
4. Search function (find similar documents)

In [ ]:
# Example 3A: Simple in-memory vector store
class SimpleVectorStore:
    def __init__(self):
        self.documents = []
        self.embeddings = []
        self.metadata = []
    
    def add_document(self, text, metadata=None):
        """Add a document to the store"""
        embedding = get_embedding(text)
        self.documents.append(text)
        self.embeddings.append(embedding)
        self.metadata.append(metadata or {})
        return len(self.documents) - 1
    
    def search(self, query, top_k=3):
        """Search for most similar documents"""
        query_embedding = get_embedding(query)
        
        # Calculate similarities
        similarities = []
        for i, doc_embedding in enumerate(self.embeddings):
            similarity = cosine_similarity(query_embedding, doc_embedding)
            similarities.append({
                'index': i,
                'document': self.documents[i],
                'similarity': similarity,
                'metadata': self.metadata[i]
            })
        
        # Sort by similarity
        similarities.sort(key=lambda x: x['similarity'], reverse=True)
        return similarities[:top_k]
    
    def get_stats(self):
        """Get store statistics"""
        return {
            'total_documents': len(self.documents),
            'embedding_dimensions': len(self.embeddings[0]) if self.embeddings else 0
        }

# Create a vector store
vector_store = SimpleVectorStore()

print("Creating vector store...\n")
print("✅ Vector store initialized!")

In [ ]:
# Example 3B: Populate vector store with documents
documents = [
    {
        "text": "Python is a high-level programming language known for its simplicity and readability. It's widely used in web development, data science, and automation.",
        "metadata": {"topic": "programming", "language": "Python"}
    },
    {
        "text": "Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed.",
        "metadata": {"topic": "AI", "subtopic": "machine learning"}
    },
    {
        "text": "The Renaissance was a period of cultural rebirth in Europe from the 14th to 17th century, marked by advances in art, science, and philosophy.",
        "metadata": {"topic": "history", "period": "Renaissance"}
    },
    {
        "text": "Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into oxygen and glucose.",
        "metadata": {"topic": "science", "subject": "biology"}
    },
    {
        "text": "Neural networks are computing systems inspired by biological neural networks in animal brains. They're the foundation of deep learning.",
        "metadata": {"topic": "AI", "subtopic": "neural networks"}
    }
]

print("Adding documents to vector store...\n")
for doc in documents:
    idx = vector_store.add_document(doc["text"], doc["metadata"])
    print(f"✓ Added document {idx}: {doc['text'][:50]}...")

stats = vector_store.get_stats()
print(f"\n📊 Store stats:")
print(f"   Documents: {stats['total_documents']}")
print(f"   Dimensions: {stats['embedding_dimensions']}")

In [ ]:
# Example 3C: Search the vector store
queries = [
    "How do plants produce food?",
    "What is deep learning?",
    "Tell me about coding languages"
]

for query in queries:
    print(f"\n{'='*80}")
    print(f"🔍 Query: {query}\n")
    
    results = vector_store.search(query, top_k=2)
    
    print("Top matches:")
    for i, result in enumerate(results, 1):
        print(f"\n  {i}. Similarity: {result['similarity']:.3f}")
        print(f"     Topic: {result['metadata'].get('topic', 'N/A')}")
        print(f"     Text: {result['document'][:80]}...")

## 4. Basic RAG (Retrieval Augmented Generation)

**RAG = Retrieval + Generation**

**How it works:**
1. **Retrieve**: Find relevant documents from vector store
2. **Augment**: Add documents to the prompt as context
3. **Generate**: AI answers using the retrieved context

**Analogy:** Like an open-book exam:
- **Regular AI**: Closed-book (only knows what it memorized)
- **RAG**: Open-book (can reference specific documents)

**Benefits:**
- Answer questions about YOUR data
- Reduce hallucinations (AI making things up)
- Keep knowledge up-to-date without retraining

In [ ]:
# Example 4A: Simple RAG implementation
def rag_query(query, vector_store, top_k=2):
    """Answer a query using RAG"""
    
    # Step 1: Retrieve relevant documents
    print(f"\n[Step 1] Retrieving relevant documents...")
    results = vector_store.search(query, top_k=top_k)
    
    # Build context from retrieved documents
    context = "\n\n".join([
        f"Document {i+1}: {r['document']}" 
        for i, r in enumerate(results)
    ])
    
    print(f"✓ Retrieved {len(results)} documents\n")
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['similarity']:.3f}] {r['document'][:60]}...")
    
    # Step 2: Augment prompt with context
    print(f"\n[Step 2] Building prompt with context...")
    prompt = f"""Answer the question based on the context below.

Context:
{context}

Question: {query}

Answer:"""
    
    # Step 3: Generate answer
    print(f"\n[Step 3] Generating answer...\n")
    response = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer based on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    
    answer = response.choices[0].message.content
    
    return {
        'answer': answer,
        'sources': results,
        'context': context
    }

# Test RAG
print("=" * 80)
print("RAG SYSTEM TEST")
print("=" * 80)

query = "What is machine learning and how does it relate to neural networks?"
print(f"\n🧑 Question: {query}")

result = rag_query(query, vector_store)

print(f"\n🤖 Answer:\n{result['answer']}")
print(f"\n📚 Sources used: {len(result['sources'])} documents")

In [ ]:
# Example 4B: RAG with source citations
def rag_with_citations(query, vector_store, top_k=3):
    """RAG that includes citations"""
    
    # Retrieve
    results = vector_store.search(query, top_k=top_k)
    
    # Build numbered context
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(f"[{i}] {r['document']}")
    
    context = "\n\n".join(context_parts)
    
    # Generate with citation instruction
    prompt = f"""Answer the question using the context below. 
Cite your sources using [1], [2], etc.

Context:
{context}

Question: {query}

Answer with citations:"""
    
    response = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Always cite sources."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    
    return {
        'answer': response.choices[0].message.content,
        'sources': results
    }

# Test with citations
print("\n" + "=" * 80)
print("RAG WITH CITATIONS")
print("=" * 80)

query = "How do computers learn from data?"
print(f"\n🧑 Question: {query}\n")

result = rag_with_citations(query, vector_store)

print(f"🤖 Answer:\n{result['answer']}")

print(f"\n📚 Sources:")
for i, source in enumerate(result['sources'], 1):
    print(f"\n[{i}] (Similarity: {source['similarity']:.3f})")
    print(f"    {source['document'][:100]}...")

In [ ]:
# Example 4C: Enhanced RAG class
class RAGSystem:
    def __init__(self, vector_store):
        self.vector_store = vector_store
        self.conversation_history = []
    
    def query(self, question, top_k=3, include_history=False):
        """Ask a question using RAG"""
        
        # Retrieve relevant docs
        results = self.vector_store.search(question, top_k=top_k)
        
        # Build context
        context = "\n\n".join([
            f"[Source {i+1}] {r['document']}"
            for i, r in enumerate(results)
        ])
        
        # Build messages
        messages = [
            {"role": "system", "content": """You are a helpful assistant. 
Answer questions based on the provided context. 
If the context doesn't contain relevant information, say so."""}
        ]
        
        # Add conversation history if requested
        if include_history:
            messages.extend(self.conversation_history)
        
        # Add current question with context
        user_message = f"""Context:
{context}

Question: {question}"""
        messages.append({"role": "user", "content": user_message})
        
        # Generate
        response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=messages,
            temperature=0.3
        )
        
        answer = response.choices[0].message.content
        
        # Save to history
        self.conversation_history.append({"role": "user", "content": question})
        self.conversation_history.append({"role": "assistant", "content": answer})
        
        return {
            'answer': answer,
            'sources': results,
            'relevance_scores': [r['similarity'] for r in results]
        }
    
    def clear_history(self):
        """Clear conversation history"""
        self.conversation_history = []

# Create RAG system
rag_system = RAGSystem(vector_store)

print("=" * 80)
print("ENHANCED RAG SYSTEM")

# Multi-turn conversation
questions = [
    "What programming language is mentioned?",
    "What is it used for?"
]

for question in questions:
    print("\n" + "=" * 80)
    print(f"🧑 Question: {question}")
    result = rag_system.query(question, top_k=2, include_history=True)
    print(f"\n🤖 Answer:\n{result['answer']}")
    print("\n📚 Sources:")
    for i, src in enumerate(result['sources'], 1):
        print(f"\n[{i}] (Similarity: {src['similarity']:.3f})")
        print(f"    {src['document'][:100]}...")
